# Overture — attribute pushdown with `where=` (live)

The bbox limits *where* you fetch; `where=` limits *which rows*, pushed down to the GeoParquet on S3 via DuckDB so only matching features leave the bucket. Here we keep only high-confidence places and a narrow set of columns (`id`/`sources` are retained automatically so the per-row `license_id` still works).

In [1]:
LAT_LIM = [40.757, 40.759]  # [south, north]
LON_LIM = [-73.987, -73.984]  # [west, east] — a Times Square block
OUT = "_overture_out"

In [2]:
from earthlens.earthlens import EarthLens

paths = EarthLens(
    data_source="overture",
    variables={"places": []},
    lat_lim=LAT_LIM, lon_lim=LON_LIM, path=OUT,
    where="confidence > 0.95",      # pushed down to S3
    columns=["names", "confidence"],  # narrow projection
).download()
paths

2026-05-27 19:31:29 | INFO | pyramids.base.config | Logging is configured.


2026-05-27 19:31:29.792 | INFO     | earthlens.overture.backend:_fetch:434 - Querying Overture 'place' (theme 'places') via DuckDB for bbox (-73.987, 40.757, -73.984, 40.759) (release=2026-05-20.0, where='confidence > 0.95')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-05-27 19:31:39.562 | INFO     | earthlens.overture.backend:_fetch:472 - places/place: wrote 95 feature(s) to C:\gdrive\algorithms\remote-sensing\earthlens\.claude\worktrees\overture\docs\examples\overture\_overture_out\overture_places_place_latest.parquet


[WindowsPath('C:/gdrive/algorithms/remote-sensing/earthlens/.claude/worktrees/overture/docs/examples/overture/_overture_out/overture_places_place_latest.parquet')]

In [3]:
import geopandas as gpd

gdf = gpd.read_parquet(paths[0])
print(len(gdf), 'high-confidence places')
print('all match the predicate:', bool((gdf['confidence'] > 0.95).all()))
gdf[['confidence', 'license_id']].head()

95 high-confidence places
all match the predicate: True


,confidence,license_id
0,0.998601,CDLA-Permissive-2.0
1,0.981101,CDLA-Permissive-2.0
2,0.990219,CDLA-Permissive-2.0
3,0.964439,CDLA-Permissive-2.0
4,0.972216,CDLA-Permissive-2.0


`where=` is raw SQL evaluated by DuckDB against the parquet schema, so you can filter nested fields too — e.g. `categories.primary = 'restaurant'` for places, or `height > 10` for buildings.